<a href="https://colab.research.google.com/github/StayWithCris/VsCode/blob/main/RetinaScan_Lab02%3A.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# RetinaScan — Lab 02
### Diagnóstico Automatizado de Retinopatía Diabética mediante CNN

**Inteligencia Artificial · 2026-I · L2-S01**

---

## Objetivo del Sistema

Este sistema automatizado procesa imágenes de fondo de ojo (retinografías) para clasificar el grado de Retinopatía Diabética (RD) en cinco niveles de severidad, basándose en la escala clínica internacional ETDRS:

| Grado | Severidad | Descripción |
|---|---|---|
| **0** | Sin RD | Retina sana |
| **1** | Leve | Microaneurismas aislados |
| **2** | Moderada | Hemorragias en puntos múltiples |
| **3** | Severa | Hemorragias extensas, exudados |
| **4** | Proliferativa | Neovascularización (riesgo de ceguera) |

El modelo integra este diagnóstico visual con parámetros clínicos del paciente (tales como HbA1c, años de diagnóstico de diabetes, presión arterial y niveles de creatinina) para calcular un índice predictivo del riesgo de ceguera a 2 años.

---

## Pipeline de Desarrollo en 7 Fases

| # | Fase | Objetivo Técnico |
|---|---|---|
| F1 | Adquisición y Análisis Exploratorio de Datos (EDA) | Carga del dataset y análisis de la distribución de las clases. |
| F2 | Preprocesamiento Retinal | Implementación de CLAHE y método de Ben Graham, con redimensionamiento a 380x380 px. |
| F3 | Aumento de Datos y Balanceo | Aplicación de rotaciones, flips horizontales y técnicas de sobremuestreo (SMOTE) para mitigar el desbalance de clases minoritarias. |
| F4 | Fine-tuning de EfficientNet-B4 | Ejecución en dos etapas: extracción de características (feature extraction) seguida de descongelamiento (unfreeze) de las capas superiores. |
| F5 | Interpretabilidad (Grad-CAM++) | Localización visual de las características (lesiones) determinantes en la clasificación. |
| F6 | Modelo MLP para Riesgo Clínico | Fusión de las características extraídas por la CNN con las variables clínicas del paciente. |
| F7 | Validación | Evaluación del rendimiento mediante sensibilidad, especificidad, índice Kappa de Cohen y matriz de confusión. |

---

> **IMPORTANTE:** Para la correcta ejecución, es indispensable habilitar la aceleración por hardware (GPU).
> Ruta: `Entorno de ejecución → Cambiar tipo de entorno → T4 GPU`.

> **Nota sobre los datasets:** El presente notebook emplea imágenes de retina sintéticas, las cuales han sido calibradas para replicar las características del dataset APTOS 2019. Los módulos de carga (loaders) para los conjuntos de datos reales se encuentran implementados al final de la sección correspondiente a la Fase 1. El uso de datos reales desde Kaggle requiere únicamente la modificación de la ruta de acceso especificada.

## Paso 0 — Verificar GPU

Las CNN son mucho más pesadas que las LSTM del Lab 1. **Necesitas GPU sí o sí.**
Con T4 GPU el entrenamiento toma ~10-15 min. Sin GPU puede tardar más de 2 horas.

In [ ]:
# @title
import subprocess, sys

# Verificar GPU disponible
resultado = subprocess.run(['nvidia-smi'], capture_output=True, text=True)

if resultado.returncode == 0:
    print(' GPU disponible:')
    # Mostrar solo las primeras líneas con info útil
    lineas = resultado.stdout.split('\n')
    for linea in lineas[:12]:
        print(linea)
else:
    print('  Sin GPU detectada.')
    print('   Ve a: Entorno de ejecución → Cambiar tipo de entorno → T4 GPU')
    print('   Esto es CRÍTICO para este lab.')

print(f'\nPython {sys.version}')


## Paso 1 — Instalar dependencias

Colab ya viene con la mayoría preinstalado. Solo necesitamos `tf-explain` para Grad-CAM++ y `albumentations` para augmentación específica de retina.

In [ ]:
%%capture
import subprocess, sys

# Solo instalamos lo que no viene por defecto en Colab
paquetes = ['tf-explain', 'albumentations']
for pkg in paquetes:
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', pkg], check=True)

print('Instalación completa ✓')


In [ ]:
# Verificar versiones de las librerías clave
import tensorflow as tf
import cv2
import sklearn
import albumentations as A

print(f'TensorFlow  : {tf.__version__}')
print(f'OpenCV      : {cv2.__version__}')
print(f'scikit-learn: {sklearn.__version__}')
print(f'Albumentations: {A.__version__}')
print(f'GPU         : {tf.config.list_physical_devices("GPU")}')


## Paso 2 — Imports y configuración global

Aquí definimos los hiperparámetros del proyecto. Si quieres experimentar con valores distintos, solo cambia los valores acá:

| Parámetro | Valor | Qué significa |
|---|---|---|
| `IMG_SIZE` | 224 | Resolución de las imágenes (224×224). El enunciado pide 380 pero 224 entrena 3× más rápido |
| `NUM_CLASSES` | 5 | Grados ETDRS: 0, 1, 2, 3, 4 |
| `BATCH_SIZE` | 32 | Imágenes procesadas en paralelo |
| `EPOCHS_HEAD` | 8 | Etapa 1: entrenar solo la cabeza clasificadora |
| `EPOCHS_FINE` | 12 | Etapa 2: descongelar últimos bloques |
| `N_TRAIN` | 3500 | Imágenes sintéticas para entrenamiento |
| `N_VAL` | 750 | Imágenes para validación |
| `N_TEST` | 750 | Imágenes para test final |

In [ ]:
import warnings
warnings.filterwarnings('ignore')

# ── Librerías estándar ─────────────────────────────────────────────────────
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import seaborn as sns
from pathlib import Path

# ── Procesamiento de imágenes ──────────────────────────────────────────────
import cv2
from PIL import Image
import albumentations as A

# ── Machine Learning clásico ───────────────────────────────────────────────
from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    classification_report, confusion_matrix, cohen_kappa_score,
    roc_auc_score, roc_curve, brier_score_loss
)
from sklearn.utils.class_weight import compute_class_weight

# ── Deep Learning ──────────────────────────────────────────────────────────
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers, callbacks, optimizers
from tensorflow.keras.applications import EfficientNetB0  # B4 es muy pesado para Colab gratuito
from tensorflow.keras.applications.efficientnet import preprocess_input

# ── Reproducibilidad ───────────────────────────────────────────────────────
SEED = 42
np.random.seed(SEED)
tf.random.set_seed(SEED)

# ── Hiperparámetros globales ───────────────────────────────────────────────
IMG_SIZE     = 224           # 224×224 para entrenamiento rápido en Colab
NUM_CLASSES  = 5             # 0=Sin RD, 1=Leve, 2=Moderada, 3=Severa, 4=Proliferativa
BATCH_SIZE   = 32
EPOCHS_HEAD  = 8             # Etapa 1: cabeza clasificadora
EPOCHS_FINE  = 12            # Etapa 2: fine-tuning
N_TRAIN      = 3500
N_VAL        = 750
N_TEST       = 750

# Nombres de las clases ETDRS
CLASS_NAMES = ['Sin RD', 'Leve', 'Moderada', 'Severa', 'Proliferativa']

# Distribución real de clases en EyePACS/APTOS (desbalance fuerte)
CLASS_DIST = [0.73, 0.07, 0.15, 0.03, 0.02]

# Colores para gráficos por clase (de verde a rojo según severidad)
CLASS_COLORS = ['#2E7D32', '#9CCC65', '#FFB300', '#F57C00', '#C62828']

print('Configuración global lista ✓')
print(f'  Imágenes: {IMG_SIZE}×{IMG_SIZE}')
print(f'  Clases  : {NUM_CLASSES} (escala ETDRS)')
print(f'  Split   : Train={N_TRAIN} / Val={N_VAL} / Test={N_TEST}')


---
# FASE 1 — Adquisición de Datos y Análisis Exploratorio (EDA)

### ¿Qué hacemos en esta fase?

1. **Generar dataset sintético** de retinografías (porque no tenemos acceso a Kaggle todavía)
2. **Analizar la distribución de clases** y confirmar el desbalance (clase 0 = 73%)
3. **Visualizar ejemplos** de cada grado de RD
4. **Preparar loaders** para datos reales (APTOS, EyePACS)

### Sobre el dataset sintético

Las imágenes simulan retinografías reales con:
- Disco óptico (círculo claro)
- Mácula central oscura
- Vasos sanguíneos ramificados
- Lesiones según el grado: microaneurismas, hemorragias, exudados

Esto permite que la CNN aprenda patrones visuales similares a los reales, y que Grad-CAM++ tenga lesiones reales que localizar.

In [ ]:
def generar_retinografia_sintetica(grado, img_size=IMG_SIZE):
    """
    Genera una imagen sintética de retinografía con apariencia médica realista.

    Componentes anatómicos simulados:
    - Fondo retinal (color rojizo-naranja característico)
    - Disco óptico (estructura clara, ovalada)
    - Mácula central (zona oscura)
    - Vasos sanguíneos (estructuras ramificadas)
    - Lesiones según el grado de severidad

    Parámetros:
        grado     : int (0-4) — severidad de retinopatía
        img_size  : int — resolución (224 o 380)

    Retorna:
        np.array RGB de forma (img_size, img_size, 3) en uint8
    """
    # ── 1. Fondo retinal (color rojizo característico) ────────────────────────
    img = np.zeros((img_size, img_size, 3), dtype=np.float32)
    centro = img_size // 2
    radio_retina = img_size // 2 - 10

    # Crear máscara circular de la retina
    Y, X = np.ogrid[:img_size, :img_size]
    dist_centro = np.sqrt((X - centro)**2 + (Y - centro)**2)
    mascara_retina = dist_centro <= radio_retina

    # Gradiente radial: más oscuro hacia los bordes
    gradiente = 1.0 - 0.4 * (dist_centro / radio_retina)
    gradiente = np.clip(gradiente, 0, 1)

    # Color base: rojo-naranja con variación
    img[..., 0] = (140 + np.random.uniform(-15, 15)) * gradiente   # R
    img[..., 1] = (75  + np.random.uniform(-10, 10)) * gradiente   # G
    img[..., 2] = (40  + np.random.uniform(-8, 8))   * gradiente   # B

    # Aplicar máscara circular (afuera queda negro)
    img[~mascara_retina] = 0

    # ── 2. Disco óptico (estructura clara) ────────────────────────────────────
    # Está hacia un lado (típicamente nasal)
    od_x = centro + int(radio_retina * 0.4 * np.random.choice([-1, 1]))
    od_y = centro + int(np.random.uniform(-20, 20))
    od_radio = int(img_size * 0.06)
    cv2.circle(img, (od_x, od_y), od_radio, (220, 200, 160), -1)
    # Difuminar el borde
    img = cv2.GaussianBlur(img, (5, 5), 0)

    # ── 3. Mácula central (zona oscura) ───────────────────────────────────────
    macula_radio = int(img_size * 0.05)
    overlay = img.copy()
    cv2.circle(overlay, (centro, centro), macula_radio, (60, 30, 15), -1)
    img = cv2.addWeighted(overlay, 0.4, img, 0.6, 0)

    # ── 4. Vasos sanguíneos (ramificaciones desde el disco óptico) ────────────
    n_vasos_principales = 6
    for i in range(n_vasos_principales):
        angulo = 2 * np.pi * i / n_vasos_principales + np.random.uniform(-0.3, 0.3)
        # Vaso principal
        x_actual, y_actual = od_x, od_y
        for paso in range(15):
            # Curvar el vaso de manera natural
            angulo += np.random.uniform(-0.15, 0.15)
            longitud = np.random.uniform(8, 15)
            x_nuevo = int(x_actual + longitud * np.cos(angulo))
            y_nuevo = int(y_actual + longitud * np.sin(angulo))
            grosor = max(1, 3 - paso // 5)
            color_vaso = (60, 20, 10) if paso < 8 else (80, 30, 15)
            cv2.line(img, (x_actual, y_actual), (x_nuevo, y_nuevo), color_vaso, grosor)
            x_actual, y_actual = x_nuevo, y_nuevo
            # Verificar que sigamos dentro de la retina
            if np.sqrt((x_nuevo-centro)**2 + (y_nuevo-centro)**2) > radio_retina - 5:
                break

    # ── 5. Lesiones según el grado de RD ──────────────────────────────────────
    # Más alta la severidad, más lesiones y de tipos más serios
    n_microaneurismas = {0: 0, 1: 3, 2: 8, 3: 15, 4: 20}[grado]
    n_hemorragias    = {0: 0, 1: 0, 2: 3, 3: 8,  4: 12}[grado]
    n_exudados       = {0: 0, 1: 0, 2: 2, 3: 6,  4: 10}[grado]
    n_neovascular    = {0: 0, 1: 0, 2: 0, 3: 1,  4: 4}[grado]

    # Microaneurismas: puntos rojos pequeños
    for _ in range(n_microaneurismas):
        x = np.random.randint(15, img_size - 15)
        y = np.random.randint(15, img_size - 15)
        if np.sqrt((x-centro)**2 + (y-centro)**2) < radio_retina - 10:
            cv2.circle(img, (x, y), np.random.randint(1, 3), (40, 20, 15), -1)

    # Hemorragias: manchas rojas más grandes
    for _ in range(n_hemorragias):
        x = np.random.randint(20, img_size - 20)
        y = np.random.randint(20, img_size - 20)
        if np.sqrt((x-centro)**2 + (y-centro)**2) < radio_retina - 15:
            radio = np.random.randint(3, 6)
            cv2.circle(img, (x, y), radio, (30, 15, 10), -1)

    # Exudados duros: puntos amarillos brillantes
    for _ in range(n_exudados):
        x = np.random.randint(20, img_size - 20)
        y = np.random.randint(20, img_size - 20)
        if np.sqrt((x-centro)**2 + (y-centro)**2) < radio_retina - 15:
            radio = np.random.randint(2, 4)
            cv2.circle(img, (x, y), radio, (240, 220, 80), -1)

    # Neovascularización: estructuras vasculares anómalas (proliferativa)
    for _ in range(n_neovascular):
        x_inicio = np.random.randint(30, img_size - 30)
        y_inicio = np.random.randint(30, img_size - 30)
        if np.sqrt((x_inicio-centro)**2 + (y_inicio-centro)**2) < radio_retina - 20:
            # Dibujar ramificación caótica
            for _ in range(8):
                dx = np.random.randint(-15, 15)
                dy = np.random.randint(-15, 15)
                cv2.line(img, (x_inicio, y_inicio),
                         (x_inicio+dx, y_inicio+dy), (50, 25, 20), 1)

    # ── 6. Ruido leve para realismo ───────────────────────────────────────────
    ruido = np.random.normal(0, 3, img.shape)
    img = np.clip(img + ruido, 0, 255).astype(np.uint8)

    return img


print('Generador de retinografías sintéticas definido ✓')


In [ ]:
# ── Generar el dataset completo ───────────────────────────────────────────
def generar_dataset(n_imagenes, dist_clases=CLASS_DIST):
    """
    Genera n_imagenes con la distribución de clases especificada.
    Replica el desbalance real de EyePACS/APTOS (73% clase 0).
    """
    n_por_clase = [int(n_imagenes * p) for p in dist_clases]
    # Ajustar para que sumen exactamente n_imagenes
    n_por_clase[0] += n_imagenes - sum(n_por_clase)

    X, y = [], []
    for clase, n in enumerate(n_por_clase):
        for _ in range(n):
            X.append(generar_retinografia_sintetica(clase))
            y.append(clase)

    # Mezclar el orden
    indices = np.random.permutation(len(X))
    X = np.array(X)[indices]
    y = np.array(y)[indices]
    return X, y


print('Generando dataset sintético (esto toma ~1-2 minutos)...')
print()

# Generar splits
X_train, y_train = generar_dataset(N_TRAIN)
print(f'  Train: {X_train.shape}  ✓')

X_val, y_val = generar_dataset(N_VAL)
print(f'  Val  : {X_val.shape}  ✓')

X_test, y_test = generar_dataset(N_TEST)
print(f'  Test : {X_test.shape}  ✓')

print('\nDataset generado exitosamente ✓')


In [ ]:
# ── Análisis de distribución de clases ────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(15, 5))
fig.suptitle('F1 · Análisis de Distribución de Clases — Replica del desbalance real EyePACS/APTOS',
             fontsize=14, fontweight='bold')

# Gráfico 1: Barras con la distribución
for nombre_split, y_split, ax in [('Train', y_train, axes[0])]:
    valores, conteos = np.unique(y_split, return_counts=True)
    porcentajes = conteos / len(y_split) * 100
    bars = ax.bar([CLASS_NAMES[v] for v in valores], conteos, color=CLASS_COLORS)
    for bar, conteo, pct in zip(bars, conteos, porcentajes):
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 30,
                f'{conteo}\n({pct:.1f}%)',
                ha='center', fontsize=10, fontweight='bold')
    ax.set_title(f'Distribución en Train ({len(y_split)} imágenes)')
    ax.set_ylabel('Cantidad de imágenes')
    ax.set_ylim(0, max(conteos) * 1.2)
    ax.grid(axis='y', alpha=0.3)

# Gráfico 2: Comparación con distribución real
ax = axes[1]
x_pos = np.arange(len(CLASS_NAMES))
width = 0.35
real_dist = np.array(CLASS_DIST) * 100
nuestro_dist = np.array([np.sum(y_train == i) for i in range(NUM_CLASSES)]) / len(y_train) * 100
ax.bar(x_pos - width/2, real_dist, width, label='Real (EyePACS/APTOS)',
       color='#1976D2', alpha=0.7)
ax.bar(x_pos + width/2, nuestro_dist, width, label='Nuestro dataset sintético',
       color='#E53935', alpha=0.7)
ax.set_xticks(x_pos)
ax.set_xticklabels(CLASS_NAMES, rotation=15)
ax.set_ylabel('% del dataset')
ax.set_title('Comparación con distribución real')
ax.legend()
ax.grid(axis='y', alpha=0.3)

plt.tight_layout()
plt.savefig('f1_distribucion_clases.png', dpi=150, bbox_inches='tight')
plt.show()

# Imprimir resumen numérico
print('\n Resumen del desbalance de clases:')
for i, nombre in enumerate(CLASS_NAMES):
    n = np.sum(y_train == i)
    pct = n / len(y_train) * 100
    print(f'  Grado {i} ({nombre:15s}): {n:5d} imágenes ({pct:5.2f}%)')

print('\n  Observación clave: la clase 0 (Sin RD) domina con ~73%.')
print('    Esto requiere weighted loss y oversampling (F3) para evitar sesgo.')


In [ ]:
# ── Visualizar ejemplos de cada grado ─────────────────────────────────────
fig, axes = plt.subplots(NUM_CLASSES, 5, figsize=(15, 3*NUM_CLASSES))
fig.suptitle('F1 · Ejemplos visuales por grado de Retinopatía Diabética',
             fontsize=14, fontweight='bold')

for clase in range(NUM_CLASSES):
    # Tomar 5 imágenes random de esa clase
    indices_clase = np.where(y_train == clase)[0]
    if len(indices_clase) >= 5:
        muestras = np.random.choice(indices_clase, 5, replace=False)
    else:
        muestras = indices_clase

    for i, idx in enumerate(muestras):
        ax = axes[clase, i]
        ax.imshow(X_train[idx])
        ax.axis('off')
        if i == 0:
            ax.set_ylabel(f'Grado {clase}\n{CLASS_NAMES[clase]}',
                          rotation=0, ha='right', va='center', fontsize=11,
                          fontweight='bold', color=CLASS_COLORS[clase])

plt.tight_layout()
plt.savefig('f1_ejemplos_por_grado.png', dpi=150, bbox_inches='tight')
plt.show()

print('\n Observa cómo varía la cantidad de lesiones según el grado:')
print('   - Grado 0: retina limpia, sin lesiones')
print('   - Grado 1: pocos microaneurismas (puntos rojos pequeños)')
print('   - Grado 2: hemorragias y exudados visibles')
print('   - Grado 3: lesiones extensas')
print('   - Grado 4: neovascularización (ramificaciones anómalas)')


In [ ]:
# ═══════════════════════════════════════════════════════════════════════════
# LOADERS PARA DATOS REALES (APTOS / EyePACS) — listos para cuando tengas Kaggle
# ═══════════════════════════════════════════════════════════════════════════

def cargar_aptos_2019(ruta_csv='train.csv', ruta_imagenes='train_images/',
                      img_size=IMG_SIZE, max_imagenes=None):
    """
    Carga el dataset APTOS 2019 (Kaggle: aptos2019-blindness-detection).

    Cómo obtenerlo:
      1. Crear cuenta en kaggle.com (gratis, con email universitario)
      2. Aceptar las reglas de la competencia en:
         kaggle.com/c/aptos2019-blindness-detection/rules
      3. En Colab:
         !pip install kaggle
         from google.colab import files
         files.upload()  # subir kaggle.json
         !mkdir -p ~/.kaggle && mv kaggle.json ~/.kaggle/ && chmod 600 ~/.kaggle/kaggle.json
         !kaggle competitions download -c aptos2019-blindness-detection
         !unzip aptos2019-blindness-detection.zip
      4. Reemplazar X_train,y_train por lo que retorna esta función:
         X_train, y_train = cargar_aptos_2019('train.csv', 'train_images/')

    Parámetros:
        ruta_csv      : ruta al CSV con id_code,diagnosis
        ruta_imagenes : carpeta con las imágenes .png
        img_size      : resolución objetivo
        max_imagenes  : limitar cantidad (None = todas)

    Retorna: X (n, img_size, img_size, 3), y (n,)
    """
    df = pd.read_csv(ruta_csv)
    if max_imagenes:
        df = df.sample(max_imagenes, random_state=SEED)

    X, y = [], []
    for _, row in df.iterrows():
        img_path = os.path.join(ruta_imagenes, f"{row['id_code']}.png")
        img = cv2.imread(img_path)
        if img is None:
            continue
        img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
        img = cv2.resize(img, (img_size, img_size))
        X.append(img)
        y.append(int(row['diagnosis']))

    return np.array(X), np.array(y)


def cargar_eyepacs(ruta_csv='trainLabels.csv', ruta_imagenes='train/',
                   img_size=IMG_SIZE, max_imagenes=None):
    """
    Carga el dataset EyePACS (Kaggle: diabetic-retinopathy-detection).

    Mismo formato que APTOS pero con 88,000+ imágenes (21 GB).
    Recomendado: usar max_imagenes=5000 para entrenamiento más rápido.

    Columnas del CSV: image,level
    """
    df = pd.read_csv(ruta_csv)
    if max_imagenes:
        df = df.sample(max_imagenes, random_state=SEED)

    X, y = [], []
    for _, row in df.iterrows():
        img_path = os.path.join(ruta_imagenes, f"{row['image']}.jpeg")
        img = cv2.imread(img_path)
        if img is None:
            continue
        img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
        img = cv2.resize(img, (img_size, img_size))
        X.append(img)
        y.append(int(row['level']))

    return np.array(X), np.array(y)


print('Loaders para datos reales definidos ✓')
print('  → cargar_aptos_2019() — APTOS 2019 (3,662 imágenes, 600 MB)')
print('  → cargar_eyepacs()    — EyePACS (88,000 imágenes, 21 GB)')
print()
print(' Para usar datos reales, sigue las instrucciones del docstring de cada función.')


---
# FASE 2 — Preprocesamiento Retinal

### ¿Por qué el preprocesamiento es crítico?

Las retinografías reales tienen problemas que dificultan que la CNN detecte lesiones:
- **Iluminación desigual** entre los bordes y el centro
- **Bajo contraste** en lesiones tenues (microaneurismas)
- **Variabilidad entre cámaras** distintas

Aplicamos 3 técnicas estándar de oftalmología digital:

| Técnica | Para qué sirve |
|---|---|
| **CLAHE** | Ecualización local de histograma → realza lesiones tenues sin amplificar ruido |
| **Ben Graham method** | Resta el fondo difuminado → elimina el gradiente de iluminación |
| **Normalización ImageNet** | Pone las imágenes en el formato que EfficientNet espera |

In [ ]:
def aplicar_clahe(img, clip_limit=2.0, tile_grid_size=(8, 8)):
    """
    CLAHE = Contrast Limited Adaptive Histogram Equalization.

    A diferencia de la ecualización normal, CLAHE divide la imagen en pequeños
    bloques y ecualiza cada uno por separado, con un límite de contraste para
    no amplificar el ruido.

    Es la técnica estándar en retinografía para realzar microaneurismas y
    hemorragias tenues.
    """
    # CLAHE funciona en escala de grises o por canal LAB
    # Convertimos a LAB y aplicamos solo en el canal L (luminancia)
    lab = cv2.cvtColor(img, cv2.COLOR_RGB2LAB)
    l, a, b = cv2.split(lab)

    clahe = cv2.createCLAHE(clipLimit=clip_limit, tileGridSize=tile_grid_size)
    l = clahe.apply(l)

    lab = cv2.merge((l, a, b))
    return cv2.cvtColor(lab, cv2.COLOR_LAB2RGB)


def aplicar_ben_graham(img, sigma_x=10):
    """
    Método de Ben Graham (ganador competencia Kaggle DR 2015).

    Resta una versión muy difuminada de la imagen para eliminar el gradiente
    de iluminación y realzar las características locales.

    Fórmula: resultado = imagen - difuminada(imagen) + 128
    """
    # Difuminar fuertemente con kernel grande
    blurred = cv2.GaussianBlur(img, (0, 0), sigma_x)
    # Restar el fondo difuminado y sumar 128 (gris medio)
    resultado = cv2.addWeighted(img, 4.0, blurred, -4.0, 128)
    return np.clip(resultado, 0, 255).astype(np.uint8)


def preprocesar_retinografia(img):
    """
    Pipeline completo de preprocesamiento:
    1. CLAHE para realzar lesiones tenues
    2. Ben Graham method para normalizar iluminación
    """
    img = aplicar_clahe(img)
    img = aplicar_ben_graham(img)
    return img


print('Funciones de preprocesamiento definidas ✓')


In [ ]:
# ── Visualizar el efecto de cada técnica ──────────────────────────────────
# Tomamos un ejemplo de cada grado y mostramos: original, CLAHE, Ben Graham, ambos
fig, axes = plt.subplots(NUM_CLASSES, 4, figsize=(16, 4*NUM_CLASSES))
fig.suptitle('F2 · Efecto del preprocesamiento por grado de RD',
             fontsize=14, fontweight='bold')

titulos = ['Original', 'CLAHE', 'Ben Graham', 'CLAHE + Ben Graham']

for clase in range(NUM_CLASSES):
    idx = np.where(y_train == clase)[0][0]
    img_original = X_train[idx]

    img_clahe = aplicar_clahe(img_original)
    img_bg = aplicar_ben_graham(img_original)
    img_ambos = preprocesar_retinografia(img_original)

    imagenes = [img_original, img_clahe, img_bg, img_ambos]

    for col, (img, titulo) in enumerate(zip(imagenes, titulos)):
        ax = axes[clase, col]
        ax.imshow(img)
        ax.axis('off')
        if clase == 0:
            ax.set_title(titulo, fontsize=12, fontweight='bold')
        if col == 0:
            ax.set_ylabel(f'Grado {clase}\n{CLASS_NAMES[clase]}',
                          rotation=0, ha='right', va='center',
                          fontsize=11, fontweight='bold',
                          color=CLASS_COLORS[clase])

plt.tight_layout()
plt.savefig('f2_preprocesamiento.png', dpi=150, bbox_inches='tight')
plt.show()

print('\n Observa cómo:')
print('   - CLAHE realza el contraste local sin saturar')
print('   - Ben Graham elimina el gradiente y hace las lesiones más visibles')
print('   - La combinación de ambos da el mejor resultado para CNN')


In [ ]:
# ── Aplicar preprocesamiento a todo el dataset ────────────────────────────
print('Aplicando CLAHE + Ben Graham a todas las imágenes...')
print('(Esto toma ~1-2 minutos)')

def preprocesar_dataset(X):
    return np.array([preprocesar_retinografia(img) for img in X])

X_train_pre = preprocesar_dataset(X_train)
X_val_pre = preprocesar_dataset(X_val)
X_test_pre = preprocesar_dataset(X_test)

print(f'\nPreprocesamiento completo ✓')
print(f'  Train: {X_train_pre.shape}  ({X_train_pre.dtype})')
print(f'  Val  : {X_val_pre.shape}')
print(f'  Test : {X_test_pre.shape}')


---
# FASE 3 — Augmentación y Balanceo de Clases

### Doble problema a resolver

1. **Desbalance:** la clase 0 tiene 73% de los datos. La CNN tendería a predecir siempre clase 0.
2. **Generalización:** queremos que el modelo funcione con retinas tomadas en distintas cámaras, ángulos y condiciones.

### Estrategia

**Augmentación con Albumentations** específica para retina (no aplicamos shifts horizontales agresivos porque la posición del disco óptico tiene significado clínico):

| Transformación | Por qué |
|---|---|
| Rotación ±15° | Las cámaras de retina giran ligeramente |
| Flip horizontal | El ojo derecho y el izquierdo son simétricos |
| Color jitter (brillo/contraste ±20%) | Cámaras distintas, luz distinta |
| Grid distortion suave | Deformación leve de la lente |

**Balanceo:** usamos `class_weight` en lugar de SMOTE en imágenes (que es costoso y poco realista). El modelo le da más importancia a las clases minoritarias durante el entrenamiento.

In [ ]:
# ── Definir el pipeline de augmentación ───────────────────────────────────
transformaciones_train = A.Compose([
    A.Rotate(limit=15, p=0.7),                          # rotación leve
    A.HorizontalFlip(p=0.5),                            # ojo izq/der son simétricos
    A.RandomBrightnessContrast(brightness_limit=0.2,
                                contrast_limit=0.2, p=0.5),
    A.GridDistortion(num_steps=5, distort_limit=0.1, p=0.3),
])

# Sin augmentación para val/test (queremos evaluación reproducible)
transformaciones_val = A.Compose([])

print('Pipeline de augmentación definido ✓')


In [ ]:
# ── Visualizar el efecto de la augmentación ───────────────────────────────
# Tomamos 1 imagen y le aplicamos 6 augmentaciones distintas
idx_ejemplo = np.where(y_train == 2)[0][0]  # Una imagen de grado 2
img_ejemplo = X_train_pre[idx_ejemplo]

fig, axes = plt.subplots(2, 4, figsize=(16, 8))
fig.suptitle('F3 · Ejemplos de augmentación sobre la misma imagen',
             fontsize=14, fontweight='bold')

# Imagen original (sin augmentación)
axes[0, 0].imshow(img_ejemplo)
axes[0, 0].set_title('Original (sin augmentación)', fontsize=11, fontweight='bold')
axes[0, 0].axis('off')

# 7 augmentaciones aleatorias
for i in range(7):
    fila, col = (i + 1) // 4, (i + 1) % 4
    img_aug = transformaciones_train(image=img_ejemplo)['image']
    axes[fila, col].imshow(img_aug)
    axes[fila, col].set_title(f'Augmentación #{i+1}', fontsize=10)
    axes[fila, col].axis('off')

plt.tight_layout()
plt.savefig('f3_augmentacion_ejemplos.png', dpi=150, bbox_inches='tight')
plt.show()

print(' Cada vez que la CNN ve la misma imagen, la ve ligeramente diferente.')
print('   Esto la obliga a aprender características invariantes (las lesiones reales)')
print('   en vez de memorizar pixeles específicos.')


In [ ]:
# ── Calcular class_weights para balancear el desbalance ───────────────────
# Esto le dice al modelo: 'cada error en clase 4 cuenta como 36 errores en clase 0'

class_weights = compute_class_weight(
    class_weight='balanced',
    classes=np.arange(NUM_CLASSES),
    y=y_train
)
class_weight_dict = {i: w for i, w in enumerate(class_weights)}

print(' Pesos calculados para balancear las clases:')
print(f'{"Grado":<8} {"Severidad":<15} {"% Dataset":<12} {"Peso":<10}')
print('─' * 50)
for i, nombre in enumerate(CLASS_NAMES):
    pct = np.sum(y_train == i) / len(y_train) * 100
    print(f'{i:<8} {nombre:<15} {pct:>6.2f}%     {class_weights[i]:>6.2f}')

print('\n Cuanto menor sea el % en el dataset, mayor el peso.')
print('   El modelo va a prestar más atención a las clases raras durante el entrenamiento.')


In [ ]:
# ── Crear generadores con augmentación on-the-fly ─────────────────────────
# Esto aplica augmentaciones diferentes en cada época sin guardar copias

class DataGenerator(keras.utils.Sequence):
    """
    Generador de batches con augmentación on-the-fly.

    En cada época, cada imagen se transforma de forma distinta.
    Esto multiplica efectivamente el tamaño del dataset sin usar memoria extra.
    """
    def __init__(self, X, y, batch_size=32, transformaciones=None, shuffle=True):
        self.X = X
        self.y = y
        self.batch_size = batch_size
        self.transformaciones = transformaciones
        self.shuffle = shuffle
        self.indices = np.arange(len(X))
        self.on_epoch_end()

    def __len__(self):
        return int(np.ceil(len(self.X) / self.batch_size))

    def __getitem__(self, idx):
        batch_indices = self.indices[idx * self.batch_size:
                                     (idx + 1) * self.batch_size]
        batch_X = []
        for i in batch_indices:
            img = self.X[i]
            if self.transformaciones:
                img = self.transformaciones(image=img)['image']
            # Preprocesamiento de EfficientNet (espera valores normalizados)
            img = preprocess_input(img.astype(np.float32))
            batch_X.append(img)

        batch_X = np.array(batch_X)
        batch_y = keras.utils.to_categorical(self.y[batch_indices],
                                              num_classes=NUM_CLASSES)
        return batch_X, batch_y

    def on_epoch_end(self):
        if self.shuffle:
            np.random.shuffle(self.indices)


# Crear generadores para train, val y test
gen_train = DataGenerator(X_train_pre, y_train, BATCH_SIZE,
                           transformaciones=transformaciones_train, shuffle=True)
gen_val = DataGenerator(X_val_pre, y_val, BATCH_SIZE,
                         transformaciones=None, shuffle=False)
gen_test = DataGenerator(X_test_pre, y_test, BATCH_SIZE,
                          transformaciones=None, shuffle=False)

print('Generadores creados ✓')
print(f'  Train: {len(gen_train)} batches de {BATCH_SIZE}')
print(f'  Val  : {len(gen_val)} batches')
print(f'  Test : {len(gen_test)} batches')


---
# FASE 4 — Fine-tuning de EfficientNet-B0 en 2 etapas

### ¿Por qué EfficientNet?

EfficientNet (Tan & Le, 2019) escala simultáneamente profundidad, ancho y resolución de la red. Para retinografía, EfficientNet-B4 (19M params) supera a ResNet-50 (25M params) con menos parámetros, lo que reduce el riesgo de sobreajuste con datasets de tamaño moderado.

> **Nota técnica:** El enunciado pide EfficientNet-B4 (19M params, 380×380). Para que Colab gratuito entrene en tiempo razonable, usamos **EfficientNet-B0** (5M params, 224×224) que es la versión más pequeña de la familia. La arquitectura es idéntica conceptualmente, solo cambia la escala. Cuando consigas más recursos, cambiar a B4 es trivial (`EfficientNetB4` en vez de `EfficientNetB0`).

### Estrategia de fine-tuning en 2 etapas

| Etapa | Qué hace | Por qué |
|---|---|---|
| **1. Feature extraction** | Congelamos la base (pesos ImageNet). Entrenamos solo la cabeza nueva. | Adaptamos la salida a 5 clases sin destruir lo aprendido |
| **2. Fine-tuning** | Descongelamos los últimos bloques. Entrenamos con lr bajo. | Especializamos los filtros a las características de retinografía |

### Función de pérdida

**Weighted Cross-Entropy** con los class_weights calculados antes. Esto castiga más fuerte los errores en clases minoritarias (severa, proliferativa).

In [ ]:
# ── Construir el modelo EfficientNet ──────────────────────────────────────
def construir_modelo_etapa1():
    """
    Modelo para la Etapa 1: feature extraction.
    Base congelada, solo entrena la cabeza nueva.
    """
    # Cargar EfficientNet-B0 preentrenado en ImageNet, sin la capa final
    base = EfficientNetB0(
        weights='imagenet',
        include_top=False,                  # quitamos la capa de 1000 clases de ImageNet
        input_shape=(IMG_SIZE, IMG_SIZE, 3)
    )

    # Congelar TODA la base
    base.trainable = False

    # Construir el modelo completo
    inputs = keras.Input(shape=(IMG_SIZE, IMG_SIZE, 3))
    x = base(inputs, training=False)
    x = layers.GlobalAveragePooling2D()(x)
    x = layers.BatchNormalization()(x)
    x = layers.Dropout(0.4)(x)
    x = layers.Dense(256, activation='relu', name='dense_intermedia')(x)
    x = layers.Dropout(0.3)(x)
    outputs = layers.Dense(NUM_CLASSES, activation='softmax', name='salida_clases')(x)

    modelo = keras.Model(inputs, outputs, name='RetinaScan_Etapa1')

    modelo.compile(
        optimizer=optimizers.Adam(learning_rate=1e-3),
        loss='categorical_crossentropy',
        metrics=['accuracy']
    )

    return modelo


print('Función para construir modelo definida ✓')
print()
print('Estructura del modelo:')
print('  Input (224×224×3)')
print('    ↓')
print('  EfficientNet-B0 base (5M params, CONGELADA)')
print('    ↓')
print('  GlobalAveragePooling2D')
print('    ↓')
print('  BatchNorm + Dropout(0.4)')
print('    ↓')
print('  Dense(256, relu)')
print('    ↓')
print('  Dropout(0.3)')
print('    ↓')
print('  Dense(5, softmax)  ← 5 grados ETDRS')


In [ ]:
# ── ETAPA 1: Entrenar solo la cabeza ──────────────────────────────────────
print('=' * 60)
print('ETAPA 1 · FEATURE EXTRACTION (base congelada)')
print('=' * 60)
print()

modelo = construir_modelo_etapa1()
modelo.summary(line_length=80)

# Callbacks para entrenamiento
callbacks_etapa1 = [
    callbacks.EarlyStopping(patience=4, restore_best_weights=True, verbose=1),
    callbacks.ReduceLROnPlateau(factor=0.5, patience=2, verbose=1),
]

historia_etapa1 = modelo.fit(
    gen_train,
    validation_data=gen_val,
    epochs=EPOCHS_HEAD,
    class_weight=class_weight_dict,    # ← balanceo de clases
    callbacks=callbacks_etapa1,
    verbose=1,
)

print('\n✓ Etapa 1 completada')


In [ ]:
# ── ETAPA 2: Fine-tuning con base descongelada ────────────────────────────
print('=' * 60)
print('ETAPA 2 · FINE-TUNING (descongelar últimos bloques)')
print('=' * 60)
print()

# Descongelar la base
modelo.layers[1].trainable = True

# Congelar TODAS las capas EXCEPTO las últimas 30
# Las primeras capas detectan bordes y texturas genéricos (no las tocamos).
# Las últimas detectan patrones específicos de retina (SÍ las ajustamos).
for layer in modelo.layers[1].layers[:-30]:
    layer.trainable = False

# Recompilar con learning rate MÁS BAJO
# Esto es crítico: si usamos lr alto, destruimos los pesos preentrenados
modelo.compile(
    optimizer=optimizers.Adam(learning_rate=1e-5),    # ← 100× más bajo
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

# Contar parámetros entrenables ahora
trainable_params = np.sum([np.prod(v.shape) for v in modelo.trainable_weights])
total_params = modelo.count_params()
print(f'Parámetros entrenables ahora: {trainable_params:,} de {total_params:,}')
print(f'(antes era solo la cabeza: ~330K)')
print()

# Callbacks más estrictos
callbacks_etapa2 = [
    callbacks.EarlyStopping(patience=5, restore_best_weights=True, verbose=1),
    callbacks.ReduceLROnPlateau(factor=0.3, patience=3, verbose=1),
    callbacks.ModelCheckpoint('retinascan_mejor.keras',
                               save_best_only=True, verbose=1),
]

historia_etapa2 = modelo.fit(
    gen_train,
    validation_data=gen_val,
    epochs=EPOCHS_FINE,
    class_weight=class_weight_dict,
    callbacks=callbacks_etapa2,
    verbose=1,
)

print('\n✓ Etapa 2 completada')
print('✓ Mejor modelo guardado en: retinascan_mejor.keras')


In [ ]:
# ── Visualizar curvas de entrenamiento ────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(15, 5))
fig.suptitle('F4 · Curvas de entrenamiento — Etapa 1 + Etapa 2',
             fontsize=14, fontweight='bold')

# Concatenar las historias de las 2 etapas
loss_train = historia_etapa1.history['loss'] + historia_etapa2.history['loss']
loss_val = historia_etapa1.history['val_loss'] + historia_etapa2.history['val_loss']
acc_train = historia_etapa1.history['accuracy'] + historia_etapa2.history['accuracy']
acc_val = historia_etapa1.history['val_accuracy'] + historia_etapa2.history['val_accuracy']

# Línea vertical donde empieza la Etapa 2
epoca_transicion = len(historia_etapa1.history['loss'])

# Gráfico 1: Loss
axes[0].plot(loss_train, label='Train', color='#1976D2', linewidth=2)
axes[0].plot(loss_val, label='Val', color='#E53935', linewidth=2)
axes[0].axvline(epoca_transicion - 0.5, color='gray', linestyle='--',
                 label='Inicio Etapa 2')
axes[0].set_xlabel('Época')
axes[0].set_ylabel('Loss (Categorical Crossentropy)')
axes[0].set_title('Pérdida durante entrenamiento')
axes[0].legend(); axes[0].grid(alpha=0.3)

# Gráfico 2: Accuracy
axes[1].plot(acc_train, label='Train', color='#1976D2', linewidth=2)
axes[1].plot(acc_val, label='Val', color='#E53935', linewidth=2)
axes[1].axvline(epoca_transicion - 0.5, color='gray', linestyle='--',
                 label='Inicio Etapa 2')
axes[1].set_xlabel('Época')
axes[1].set_ylabel('Accuracy')
axes[1].set_title('Exactitud durante entrenamiento')
axes[1].set_ylim(0, 1)
axes[1].legend(); axes[1].grid(alpha=0.3)

plt.tight_layout()
plt.savefig('f4_curvas_entrenamiento.png', dpi=150, bbox_inches='tight')
plt.show()

print('\n Observa cómo:')
print('   - En la Etapa 1, la val_acc sube rápido (la cabeza aprende)')
print('   - En la Etapa 2, mejora más lento pero refina los detalles')
print('   - Si val_loss sube mientras train_loss baja → estamos sobreajustando')


---
# FASE 5 — Explicabilidad con Grad-CAM++

### El problema de la 'caja negra'

Una CNN que predice 'grado 3 con 89% de probabilidad' no le sirve a un médico si no puede ver POR QUÉ.  
Necesitamos mostrar **qué zona de la retina** activó esa predicción.

### ¿Cómo funciona Grad-CAM++?

1. Toma la imagen → la pasa por la CNN → obtiene la predicción
2. Calcula los **gradientes** de la clase predicha respecto a la última capa convolucional
3. Esos gradientes indican qué regiones de la imagen **'empujaron'** la predicción
4. Genera un **mapa de calor** (rojo = importante, azul = irrelevante)
5. Se superpone sobre la imagen original

Es como preguntarle al modelo: *¿dónde estás mirando?*

In [ ]:
# ── Implementación de Grad-CAM++ ──────────────────────────────────────────
def obtener_gradcam(imagen, modelo, indice_clase=None,
                    nombre_capa_conv='top_conv'):
    """
    Genera el mapa de calor Grad-CAM para una imagen.

    Parámetros:
        imagen        : array (H, W, 3) en uint8
        modelo        : modelo Keras entrenado
        indice_clase  : clase a explicar (None = clase predicha)
        nombre_capa_conv : última capa conv del modelo

    Retorna:
        heatmap : array (H, W) con valores entre 0 y 1
        clase_predicha : int
        confianza : float
    """
    # Buscar la última capa convolucional de EfficientNet
    base_efficientnet = modelo.layers[1]
    capa_conv = None
    for layer in reversed(base_efficientnet.layers):
        if isinstance(layer, layers.Conv2D):
            capa_conv = layer
            break

    # Modelo intermedio que devuelve activaciones + predicción
    grad_model = keras.Model(
        inputs=modelo.input,
        outputs=[capa_conv.output, modelo.output]
    )

    # Preprocesar imagen
    img_tensor = preprocess_input(imagen.astype(np.float32))[np.newaxis, ...]

    # Calcular gradientes
    with tf.GradientTape() as tape:
        activaciones, predicciones = grad_model(img_tensor)
        if indice_clase is None:
            indice_clase = tf.argmax(predicciones[0])
        clase_output = predicciones[:, indice_clase]

    # Gradientes respecto a las activaciones de la última conv
    gradientes = tape.gradient(clase_output, activaciones)

    # Pooled gradients (promedio global)
    pooled_gradients = tf.reduce_mean(gradientes, axis=(0, 1, 2))

    # Multiplicar cada canal de activación por su importancia
    activaciones = activaciones[0].numpy()
    for i in range(pooled_gradients.shape[-1]):
        activaciones[..., i] *= pooled_gradients[i].numpy()

    # Promedio sobre los canales → heatmap final
    heatmap = np.mean(activaciones, axis=-1)
    heatmap = np.maximum(heatmap, 0)             # ReLU: solo valores positivos
    if heatmap.max() > 0:
        heatmap /= heatmap.max()                  # normalizar a [0, 1]

    # Redimensionar al tamaño de la imagen original
    heatmap_resized = cv2.resize(heatmap, (imagen.shape[1], imagen.shape[0]))

    return heatmap_resized, int(indice_clase), float(predicciones[0, indice_clase])


def superponer_heatmap(imagen, heatmap, alpha=0.4, colormap=cv2.COLORMAP_JET):
    """
    Superpone el heatmap sobre la imagen original con transparencia.
    """
    heatmap_uint8 = np.uint8(255 * heatmap)
    heatmap_color = cv2.applyColorMap(heatmap_uint8, colormap)
    heatmap_color = cv2.cvtColor(heatmap_color, cv2.COLOR_BGR2RGB)
    superpuesto = cv2.addWeighted(imagen, 1 - alpha, heatmap_color, alpha, 0)
    return superpuesto


print('Funciones de Grad-CAM definidas ✓')


In [ ]:
# ── Generar Grad-CAM para muestras de cada grado ──────────────────────────
# Tomamos 1 ejemplo de cada clase del conjunto de test
fig, axes = plt.subplots(NUM_CLASSES, 3, figsize=(12, 4*NUM_CLASSES))
fig.suptitle('F5 · Grad-CAM por grado de RD — ¿Dónde mira el modelo?',
             fontsize=14, fontweight='bold')

for clase in range(NUM_CLASSES):
    # Buscar una imagen del test que el modelo prediga correctamente
    indices_clase = np.where(y_test == clase)[0]
    idx = indices_clase[0]
    imagen_original = X_test_pre[idx]

    # Generar Grad-CAM
    heatmap, clase_pred, confianza = obtener_gradcam(imagen_original, modelo)
    superpuesto = superponer_heatmap(imagen_original, heatmap)

    # Columna 1: imagen preprocesada
    axes[clase, 0].imshow(imagen_original)
    axes[clase, 0].axis('off')
    if clase == 0:
        axes[clase, 0].set_title('Imagen preprocesada', fontweight='bold')

    # Columna 2: heatmap solo
    axes[clase, 1].imshow(heatmap, cmap='jet')
    axes[clase, 1].axis('off')
    if clase == 0:
        axes[clase, 1].set_title('Heatmap Grad-CAM', fontweight='bold')

    # Columna 3: superposición
    axes[clase, 2].imshow(superpuesto)
    axes[clase, 2].axis('off')
    if clase == 0:
        axes[clase, 2].set_title('Superpuesto sobre original', fontweight='bold')

    # Etiqueta de la fila
    axes[clase, 0].set_ylabel(
        f'Real: Grado {clase}\n'
        f'{CLASS_NAMES[clase]}\n\n'
        f'Pred: Grado {clase_pred}\n'
        f'{CLASS_NAMES[clase_pred]}\n'
        f'Conf: {confianza:.1%}',
        rotation=0, ha='right', va='center',
        fontsize=10, fontweight='bold',
        color=CLASS_COLORS[clase] if clase_pred == clase else 'gray'
    )

plt.tight_layout()
plt.savefig('f5_gradcam_por_grado.png', dpi=150, bbox_inches='tight')
plt.show()

print('\n Las zonas en ROJO/AMARILLO son las que más influyeron en la predicción.')
print('   Idealmente debería coincidir con las lesiones visibles:')
print('   - Grado 0: el modelo mira la retina sana')
print('   - Grados 1-2: enfoque en microaneurismas y hemorragias')
print('   - Grados 3-4: zonas con neovascularización y exudados extensos')


---
# FASE 6 — MLP de Riesgo Clínico (fusión multimodal)

### ¿Por qué necesitamos un segundo modelo?

La CNN nos dice **qué hay en la retina** (grado de RD), pero el riesgo real de ceguera depende también de:

- **HbA1c**: hemoglobina glicosilada (control de la glucosa a 3 meses)
- **Años con diabetes**: a más años, más probable la progresión
- **Presión arterial sistólica**: la hipertensión empeora la RD
- **Creatinina sérica**: marcador de función renal (los riñones y los ojos se dañan juntos)

Combinamos los **5 outputs de la CNN** (probabilidad de cada grado) con las **4 variables clínicas** = 9 features de entrada al MLP.

### Arquitectura del MLP

```
Input: 9 features (5 CNN + 4 clínicas)
    ↓
Dense(64, ReLU)
    ↓
Dropout(0.2)
    ↓
Dense(32, ReLU)
    ↓
Output: 2 valores
    - Índice de riesgo de ceguera a 2 años (0-1)
    - Prioridad de derivación (urgente/rutina/control)
```

### Datos clínicos

Generamos datos sintéticos basados en los rangos del estudio **UKPDS** (UK Prospective Diabetes Study), la referencia internacional para diabetes tipo 2.

In [ ]:
# ── Generar datos clínicos sintéticos por paciente ────────────────────────
def generar_datos_clinicos(grado_rd, n=1):
    """
    Genera variables clínicas correlacionadas con el grado de RD.

    Lógica clínica:
    - Grados altos de RD → HbA1c más alta (peor control glucémico)
    - Grados altos → más años con diabetes
    - Grados altos → presión arterial más elevada
    - Grados altos → creatinina más elevada (daño renal asociado)

    Rangos basados en UKPDS y guías ADA 2024.
    """
    # Medias y desviaciones por grado (calibradas con literatura)
    hba1c_means    = [6.2, 6.8, 7.5, 8.5, 9.5]   # %
    anos_dm_means  = [3,   7,   12,  18,  22]    # años
    pas_means      = [125, 130, 138, 145, 152]   # mmHg
    crea_means     = [0.8, 0.9, 1.1, 1.4, 1.8]   # mg/dL

    datos = []
    for _ in range(n):
        hba1c = np.clip(np.random.normal(hba1c_means[grado_rd], 0.7), 5.0, 13.0)
        anos_dm = np.clip(np.random.normal(anos_dm_means[grado_rd], 3), 0, 35)
        pas = np.clip(np.random.normal(pas_means[grado_rd], 10), 100, 200)
        crea = np.clip(np.random.normal(crea_means[grado_rd], 0.3), 0.5, 3.5)
        datos.append([hba1c, anos_dm, pas, crea])

    return np.array(datos)


def calcular_riesgo_real(grado_rd, hba1c, anos_dm, pas, crea):
    """
    Calcula el riesgo 'real' de ceguera a 2 años basado en literatura.

    Esta es la 'verdad ground truth' que el MLP va a aprender a predecir.
    En un caso real esto saldría de seguimiento longitudinal de pacientes.
    """
    # Riesgo base por grado de RD (UKPDS / DCCT)
    riesgo_base = {0: 0.01, 1: 0.05, 2: 0.15, 3: 0.45, 4: 0.75}[grado_rd]

    # Multiplicadores por factores de riesgo
    mult_hba1c = 1 + (hba1c - 7) * 0.15 if hba1c > 7 else 1
    mult_anos = 1 + (anos_dm - 10) * 0.02 if anos_dm > 10 else 1
    mult_pas = 1 + (pas - 130) * 0.005 if pas > 130 else 1
    mult_crea = 1 + (crea - 1) * 0.3 if crea > 1 else 1

    riesgo = riesgo_base * mult_hba1c * mult_anos * mult_pas * mult_crea
    return np.clip(riesgo, 0, 1)


print('Funciones clínicas definidas ✓')


In [ ]:
# ── Crear el dataset combinado para entrenar el MLP ───────────────────────
print('Generando features combinados (CNN + clínico) para entrenar el MLP...')

def crear_features_mlp(X_imagenes, y_grados, modelo_cnn):
    """
    Para cada imagen:
    1. Obtener las 5 probabilidades de la CNN
    2. Generar 4 variables clínicas sintéticas
    3. Calcular el riesgo 'real' como ground truth
    """
    # Predicciones de la CNN (probabilidades softmax)
    print('  Obteniendo predicciones CNN...')
    X_preprocesado = np.array([preprocess_input(img.astype(np.float32))
                                for img in X_imagenes])
    probs_cnn = modelo_cnn.predict(X_preprocesado, verbose=0)

    # Generar variables clínicas y riesgo real
    print('  Generando variables clínicas...')
    datos_clinicos = []
    riesgos_reales = []
    prioridades_reales = []

    for grado in y_grados:
        # 4 variables clínicas
        clinicos = generar_datos_clinicos(grado, n=1)[0]
        datos_clinicos.append(clinicos)

        # Riesgo de ceguera a 2 años
        riesgo = calcular_riesgo_real(grado, *clinicos)
        riesgos_reales.append(riesgo)

        # Prioridad de derivación:
        # 0=control anual, 1=rutina (3 meses), 2=urgente (1 mes)
        if grado >= 3 or riesgo > 0.5:
            prioridad = 2  # Urgente
        elif grado >= 2 or riesgo > 0.2:
            prioridad = 1  # Rutina
        else:
            prioridad = 0  # Control anual
        prioridades_reales.append(prioridad)

    datos_clinicos = np.array(datos_clinicos)
    riesgos_reales = np.array(riesgos_reales)
    prioridades_reales = np.array(prioridades_reales)

    # Combinar: 5 features CNN + 4 clínicos = 9 features totales
    X_mlp = np.concatenate([probs_cnn, datos_clinicos], axis=1)
    return X_mlp, riesgos_reales, prioridades_reales


# Crear datasets para el MLP
X_mlp_train, riesgo_train, prio_train = crear_features_mlp(X_train_pre, y_train, modelo)
X_mlp_val,   riesgo_val,   prio_val   = crear_features_mlp(X_val_pre,   y_val,   modelo)
X_mlp_test,  riesgo_test,  prio_test  = crear_features_mlp(X_test_pre,  y_test,  modelo)

print(f'\n✓ Features para MLP creados:')
print(f'  Train: {X_mlp_train.shape}  (9 features = 5 CNN + 4 clínicos)')
print(f'  Val  : {X_mlp_val.shape}')
print(f'  Test : {X_mlp_test.shape}')
print(f'\n  Distribución de prioridades en train:')
for p, etiqueta in enumerate(['Control anual', 'Rutina (3 meses)', 'Urgente']):
    n = np.sum(prio_train == p)
    print(f'    {etiqueta}: {n} ({n/len(prio_train)*100:.1f}%)')


In [ ]:
# ── Construir y entrenar el MLP ───────────────────────────────────────────
def construir_mlp_riesgo():
    """
    MLP de 3 capas con doble salida:
    - Salida 1 (regresión): índice de riesgo de ceguera 2 años
    - Salida 2 (clasificación): prioridad de derivación (3 clases)
    """
    inputs = keras.Input(shape=(9,), name='features_fusionados')
    x = layers.Dense(64, activation='relu', name='dense_1')(inputs)
    x = layers.Dropout(0.2)(x)
    x = layers.Dense(32, activation='relu', name='dense_2')(x)
    x = layers.Dropout(0.1)(x)

    # Salida 1: riesgo (regresión continua 0-1)
    riesgo_out = layers.Dense(1, activation='sigmoid', name='riesgo_ceguera')(x)

    # Salida 2: prioridad (clasificación 3 clases)
    prio_out = layers.Dense(3, activation='softmax', name='prioridad_derivacion')(x)

    mlp = keras.Model(inputs, [riesgo_out, prio_out], name='MLP_RiesgoClinico')

    mlp.compile(
        optimizer=optimizers.Adam(1e-3),
        loss={
            'riesgo_ceguera': 'mse',
            'prioridad_derivacion': 'sparse_categorical_crossentropy'
        },
        loss_weights={'riesgo_ceguera': 1.0, 'prioridad_derivacion': 1.0},
        metrics={
            'riesgo_ceguera': ['mae'],
            'prioridad_derivacion': ['accuracy']
        }
    )
    return mlp


mlp = construir_mlp_riesgo()
mlp.summary()

# Entrenar
print('\n' + '=' * 60)
print('Entrenando MLP de riesgo clínico...')
print('=' * 60)

historia_mlp = mlp.fit(
    X_mlp_train,
    {'riesgo_ceguera': riesgo_train, 'prioridad_derivacion': prio_train},
    validation_data=(
        X_mlp_val,
        {'riesgo_ceguera': riesgo_val, 'prioridad_derivacion': prio_val}
    ),
    epochs=30,
    batch_size=32,
    callbacks=[
        callbacks.EarlyStopping(patience=8, restore_best_weights=True),
        callbacks.ReduceLROnPlateau(factor=0.5, patience=4),
    ],
    verbose=0,
)

print('\n✓ MLP entrenado')
print(f'  Épocas: {len(historia_mlp.history["loss"])}')
print(f'  MAE final riesgo (val): {historia_mlp.history["val_riesgo_ceguera_mae"][-1]:.4f}')
print(f'  Accuracy prioridad (val): {historia_mlp.history["val_prioridad_derivacion_accuracy"][-1]:.4f}')

# Guardar MLP
mlp.save('mlp_riesgo.keras')
print('✓ MLP guardado: mlp_riesgo.keras')


In [ ]:
# ── Visualizar resultados del MLP ─────────────────────────────────────────
fig, axes = plt.subplots(2, 2, figsize=(15, 10))
fig.suptitle('F6 · Resultados del MLP de Riesgo Clínico',
             fontsize=14, fontweight='bold')

# 1. Predicciones de riesgo vs real
riesgo_pred, prio_pred = mlp.predict(X_mlp_test, verbose=0)
riesgo_pred = riesgo_pred.flatten()

axes[0, 0].scatter(riesgo_test, riesgo_pred, alpha=0.3, color='#1976D2')
axes[0, 0].plot([0, 1], [0, 1], 'r--', label='Predicción perfecta')
axes[0, 0].set_xlabel('Riesgo real')
axes[0, 0].set_ylabel('Riesgo predicho')
axes[0, 0].set_title('Predicción del riesgo de ceguera a 2 años')
axes[0, 0].legend()
axes[0, 0].grid(alpha=0.3)

# 2. Matriz de confusión de prioridad
prio_pred_clase = np.argmax(prio_pred, axis=1)
cm_prio = confusion_matrix(prio_test, prio_pred_clase)
sns.heatmap(cm_prio, annot=True, fmt='d', cmap='Blues', ax=axes[0, 1],
            xticklabels=['Control', 'Rutina', 'Urgente'],
            yticklabels=['Control', 'Rutina', 'Urgente'])
axes[0, 1].set_xlabel('Predicción')
axes[0, 1].set_ylabel('Real')
axes[0, 1].set_title('Matriz de confusión — Prioridad de derivación')

# 3. Distribución de riesgo por grado de RD
for grado in range(NUM_CLASSES):
    mask = y_test == grado
    axes[1, 0].hist(riesgo_pred[mask], bins=20, alpha=0.5,
                     label=f'Grado {grado}', color=CLASS_COLORS[grado])
axes[1, 0].set_xlabel('Riesgo predicho de ceguera 2 años')
axes[1, 0].set_ylabel('Frecuencia')
axes[1, 0].set_title('Distribución de riesgo por grado de RD')
axes[1, 0].legend()
axes[1, 0].grid(alpha=0.3)

# 4. Curvas de entrenamiento
axes[1, 1].plot(historia_mlp.history['loss'], label='Train', color='#1976D2')
axes[1, 1].plot(historia_mlp.history['val_loss'], label='Val', color='#E53935')
axes[1, 1].set_xlabel('Época')
axes[1, 1].set_ylabel('Loss total')
axes[1, 1].set_title('Curva de entrenamiento del MLP')
axes[1, 1].legend()
axes[1, 1].grid(alpha=0.3)

plt.tight_layout()
plt.savefig('f6_mlp_resultados.png', dpi=150, bbox_inches='tight')
plt.show()

print('\n El MLP aprendió a combinar el diagnóstico visual con el contexto clínico.')
print('   Esto permite que dos pacientes con el mismo grado de RD reciban')
print('   distinta prioridad si sus factores de riesgo son distintos.')


---
# FASE 7 — Validación y Métricas Clínicas

### ¿Qué medimos y por qué?

En tamizaje oftalmológico, **un falso negativo es peor que un falso positivo**:
- Falso positivo → paciente derivado innecesariamente (molesto, costoso)
- Falso negativo → paciente con RD severa que NO recibe tratamiento → ceguera irreversible

Por eso priorizamos **sensibilidad** sobre especificidad.

### Métricas objetivo según el enunciado

| Métrica | Valor objetivo | Por qué |
|---|---|---|
| **Sensibilidad RD referable (≥2)** | ≥ 0.90 | Detectar ≥90% de casos que necesitan derivación |
| **Especificidad** | ≥ 0.85 | No alarmar al 85% de pacientes sanos |
| **Kappa de Cohen** | ≥ 0.75 | Acuerdo sustancial con oftalmólogo |
| **AUC-ROC (grado 3-4)** | ≥ 0.92 | Discriminación precisa de casos severos |
| **Falsos negativos** | ≤ 10% | Error clínicamente crítico |

In [ ]:
# ── Evaluación completa del modelo en el conjunto de TEST ─────────────────
print('=' * 60)
print('EVALUACIÓN FINAL — Conjunto de Test')
print('=' * 60)

# Predicciones de la CNN
X_test_norm = np.array([preprocess_input(img.astype(np.float32)) for img in X_test_pre])
probs_cnn_test = modelo.predict(X_test_norm, verbose=0)
y_pred = np.argmax(probs_cnn_test, axis=1)

# Métricas globales
accuracy_global = np.mean(y_pred == y_test)
kappa = cohen_kappa_score(y_test, y_pred, weights='quadratic')

print(f'\n Métricas globales:')
print(f'   Accuracy total       : {accuracy_global:.4f}')
print(f'   Kappa de Cohen (cuadr.): {kappa:.4f}  (objetivo: ≥ 0.75)')

# Reporte de clasificación detallado
print(f'\n Reporte por clase:\n')
print(classification_report(y_test, y_pred, target_names=CLASS_NAMES, digits=3))

# Análisis binario: RD referable (grado ≥ 2) vs no referable (grado < 2)
y_test_binaria = (y_test >= 2).astype(int)
y_pred_binaria = (y_pred >= 2).astype(int)

# Probabilidad de ser referable (suma de probs de clases 2, 3, 4)
prob_referable = probs_cnn_test[:, 2:].sum(axis=1)

tn, fp, fn, tp = confusion_matrix(y_test_binaria, y_pred_binaria).ravel()
sensibilidad = tp / (tp + fn)
especificidad = tn / (tn + fp)
tasa_fn = fn / (tp + fn)
auroc_referable = roc_auc_score(y_test_binaria, prob_referable)

print('\n Métricas clínicas (RD referable = grado ≥ 2):')
print(f'   Sensibilidad         : {sensibilidad:.4f}  (objetivo: ≥ 0.90)')
print(f'   Especificidad        : {especificidad:.4f}  (objetivo: ≥ 0.85)')
print(f'   AUC-ROC              : {auroc_referable:.4f}  (objetivo: ≥ 0.92)')
print(f'   Tasa falsos negativos: {tasa_fn:.4f}  (objetivo: ≤ 0.10)')

# Verificar cumplimiento de objetivos
print('\n Cumplimiento de objetivos del enunciado:')
print(f'   Sensibilidad ≥ 0.90 : {"✓ SÍ" if sensibilidad >= 0.90 else "✗ NO (" + f"{sensibilidad:.3f}" + ")"}')
print(f'   Especificidad ≥ 0.85: {"✓ SÍ" if especificidad >= 0.85 else "✗ NO (" + f"{especificidad:.3f}" + ")"}')
print(f'   Kappa ≥ 0.75        : {"✓ SÍ" if kappa >= 0.75 else "✗ NO (" + f"{kappa:.3f}" + ")"}')
print(f'   AUC ≥ 0.92          : {"✓ SÍ" if auroc_referable >= 0.92 else "✗ NO (" + f"{auroc_referable:.3f}" + ")"}')
print(f'   FN ≤ 10%            : {"✓ SÍ" if tasa_fn <= 0.10 else "✗ NO (" + f"{tasa_fn:.3f}" + ")"}')


In [ ]:
# ── Visualización completa de la evaluación ───────────────────────────────
fig = plt.figure(figsize=(18, 12))
gs = gridspec.GridSpec(3, 3, figure=fig)
fig.suptitle('F7 · Evaluación Clínica Completa de RetinaScan',
             fontsize=14, fontweight='bold')

# 1. Matriz de confusión multiclase (5x5)
ax1 = fig.add_subplot(gs[0:2, 0:2])
cm = confusion_matrix(y_test, y_pred)
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=ax1,
            xticklabels=CLASS_NAMES, yticklabels=CLASS_NAMES,
            cbar_kws={'label': 'Cantidad'})
ax1.set_xlabel('Predicción')
ax1.set_ylabel('Valor real')
ax1.set_title('Matriz de Confusión (5 grados ETDRS)',
              fontsize=12, fontweight='bold')

# 2. Curva ROC RD referable
ax2 = fig.add_subplot(gs[0, 2])
fpr, tpr, _ = roc_curve(y_test_binaria, prob_referable)
ax2.plot(fpr, tpr, lw=2.5, color='#1976D2',
         label=f'AUC = {auroc_referable:.3f}')
ax2.plot([0, 1], [0, 1], 'k--', alpha=0.4, label='Aleatorio')
ax2.set_xlabel('Tasa Falsos Positivos (1-Especificidad)')
ax2.set_ylabel('Tasa Verdaderos Positivos (Sensibilidad)')
ax2.set_title('Curva ROC — RD Referable')
ax2.legend()
ax2.grid(alpha=0.3)

# 3. Bar chart de métricas vs objetivos
ax3 = fig.add_subplot(gs[1, 2])
metricas = ['Sensib.', 'Especif.', 'Kappa', 'AUC']
valores = [sensibilidad, especificidad, kappa, auroc_referable]
objetivos = [0.90, 0.85, 0.75, 0.92]
colores = ['#43A047' if v >= o else '#E53935' for v, o in zip(valores, objetivos)]

x_pos = np.arange(len(metricas))
bars = ax3.bar(x_pos, valores, color=colores, alpha=0.7)
ax3.scatter(x_pos, objetivos, color='black', s=80, marker='_',
             label='Objetivo', linewidth=3)
for bar, valor in zip(bars, valores):
    ax3.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.02,
             f'{valor:.3f}', ha='center', fontsize=9, fontweight='bold')
ax3.set_xticks(x_pos)
ax3.set_xticklabels(metricas)
ax3.set_ylim(0, 1.1)
ax3.set_title('Métricas vs Objetivos')
ax3.legend()
ax3.grid(axis='y', alpha=0.3)

# 4. AUC-ROC por clase
ax4 = fig.add_subplot(gs[2, 0])
for clase in range(NUM_CLASSES):
    y_clase = (y_test == clase).astype(int)
    if y_clase.sum() > 0:  # solo si hay ejemplos de esa clase
        fpr_c, tpr_c, _ = roc_curve(y_clase, probs_cnn_test[:, clase])
        auc_c = roc_auc_score(y_clase, probs_cnn_test[:, clase])
        ax4.plot(fpr_c, tpr_c, color=CLASS_COLORS[clase],
                 label=f'Grado {clase}: {auc_c:.3f}')
ax4.plot([0, 1], [0, 1], 'k--', alpha=0.4)
ax4.set_xlabel('FPR')
ax4.set_ylabel('TPR')
ax4.set_title('ROC por grado')
ax4.legend(fontsize=8)
ax4.grid(alpha=0.3)

# 5. Distribución de errores
ax5 = fig.add_subplot(gs[2, 1])
errores_por_clase = []
for clase in range(NUM_CLASSES):
    mask = y_test == clase
    if mask.sum() > 0:
        error = np.mean(y_pred[mask] != clase)
    else:
        error = 0
    errores_por_clase.append(error)
ax5.bar(range(NUM_CLASSES), errores_por_clase, color=CLASS_COLORS)
ax5.set_xlabel('Grado real')
ax5.set_ylabel('Tasa de error')
ax5.set_title('Errores por grado')
ax5.set_xticks(range(NUM_CLASSES))
ax5.set_xticklabels([f'G{i}' for i in range(NUM_CLASSES)])
ax5.grid(axis='y', alpha=0.3)

# 6. Resumen final
ax6 = fig.add_subplot(gs[2, 2])
ax6.axis('off')
resumen_texto = f'''
RESUMEN FINAL
─────────────
Accuracy:    {accuracy_global:.3f}
Kappa:       {kappa:.3f}
Sensib.:     {sensibilidad:.3f}
Especif.:    {especificidad:.3f}
AUC:         {auroc_referable:.3f}
FN rate:     {tasa_fn:.3f}

Test: {len(y_test)} imágenes
'''
ax6.text(0.1, 0.5, resumen_texto, family='monospace',
         fontsize=11, verticalalignment='center',
         bbox=dict(boxstyle='round', facecolor='#F5F7FA', edgecolor='#1976D2'))

plt.tight_layout()
plt.savefig('f7_evaluacion_completa.png', dpi=150, bbox_inches='tight')
plt.show()


In [ ]:
# ── Guardar todo para usar en el dashboard ────────────────────────────────
import pickle

# Guardar metadatos para el dashboard
metadata = {
    'class_names': CLASS_NAMES,
    'class_colors': CLASS_COLORS,
    'img_size': IMG_SIZE,
    'num_classes': NUM_CLASSES,
    'metrics': {
        'accuracy': float(accuracy_global),
        'kappa': float(kappa),
        'sensibilidad': float(sensibilidad),
        'especificidad': float(especificidad),
        'auc_roc': float(auroc_referable),
        'tasa_fn': float(tasa_fn),
    }
}

with open('retinascan_metadata.pkl', 'wb') as f:
    pickle.dump(metadata, f)

print('✓ Archivos generados para usar en el Dashboard Streamlit:')
print('   - retinascan_mejor.keras   (modelo CNN)')
print('   - mlp_riesgo.keras         (MLP de riesgo)')
print('   - retinascan_metadata.pkl  (metadatos y métricas)')
print()
print(' Para descargar a tu PC desde Colab, ejecuta:')
print('   from google.colab import files')
print('   files.download("retinascan_mejor.keras")')
print('   files.download("mlp_riesgo.keras")')
print('   files.download("retinascan_metadata.pkl")')


---

**Integrantes:** Ignacio Moreno y Cristóbal Faúndez · **Lab 02 · RetinaScan · IA 2026-I**